# Analyzing the drivers of performance ratings

In [3]:
# install scikit-posthocs for Dunn's post-hoc test
!pip install -q scikit-posthocs

In [4]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# statistical tests
from scipy import stats

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ordinal regression
from statsmodels.miscmodels.ordinal_model import OrderedModel

# model diagnostics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# display settings
pd.set_option("display.max_columns", None)

# plotting style
sns.set_style("whitegrid")

# import Dunn's post-hoc testing package
import scikit_posthocs as sp

**Initial observations and the key modeling question:**


*   Are performance ratings primarily driven by job-relevant performance indicators (sales, new_customers), or do contextual/demographic variables like region and gender also influence ratings?

*   Rating is not truly continuous; it is ordinal
*   The natural model is likely ordered logistic regression, not ordinary OLS

Note: sales is in millions of dollars

In [5]:
# load employee performance dataset
url = "https://raw.githubusercontent.com/keithmcnulty/peopleanalytics-regression-book/master/data/employee_performance.csv"

employee_df = pd.read_csv(url)

# basic inspection
print(employee_df.shape)
print("\nColumns:")
print(employee_df.columns)

employee_df.head()

(366, 5)

Columns:
Index(['sales', 'new_customers', 'region', 'gender', 'rating'], dtype='object')


,sales,new_customers,region,gender,rating
0,10.10,9,North,M,2
1,11.67,17,West,F,3
2,8.91,13,North,M,3
3,7.73,10,South,F,2
4,11.36,12,North,M,2


**Business / Research Framing**

Primary question:

Are employee performance ratings mainly explained by measurable performance, or are ratings also associated with non-performance factors such as region or gender?

Key stakeholder concerns:

*   Are ratings performance-based?

*   Are ratings fair?

*   Do sales and new_customers predict higher ratings?

*   Does gender matter after accounting for performance?


*   Does region matter, and if so, is there a legitimate business explanation?









Variable Classification
**bold text**

| Variable        | Type                         | Role                        |
| --------------- | ---------------------------- | --------------------------- |
| `rating`        | ordinal categorical: 1, 2, 3 | outcome                     |
| `sales`         | continuous                   | performance predictor       |
| `new_customers` | numeric/count                | performance predictor       |
| `region`        | nominal categorical          | contextual/control variable |
| `gender`        | binary categorical           | fairness/process variable   |


**Exploratory Data Analysis**

Goal:

*   Understand whether higher ratings appear descriptively related to stronger measurable performance, and whether ratings differ across demographic or regional groups.


Observations


*   The rating outcome is not severely imbalanced.

*   Rating 2 is the most common, but all three categories have enough observations for modeling.


*   Gender and region are both reasonably balanced, which is helpful for fairness/context analysis.


*   sales and new_customers both increase clearly as rating increases, which is exactly what we would hope to see if the rating system is performance-driven.


*   The correlation between sales and new_customers is about 0.40, which is moderate but not concerning for multicollinearity by itself.















The strongest early EDA signal is this:

*   Average sales and average new customers both rise monotonically from rating 1 to rating 3.

*   That supports the idea that the rating process is at least partly aligned with measurable performance.







In [6]:
# inspect data types
employee_df.dtypes

,0
sales,float64
new_customers,int64
region,object
gender,object
rating,int64


In [7]:
# check missing values
employee_df.isnull().sum()

,0
sales,0
new_customers,0
region,0
gender,0
rating,0


In [8]:
# summary statistics
employee_df.describe()

,sales,new_customers,rating
count,366.000000,366.000000,366.000000
mean,7.543142,8.024590,1.937158
std,2.934558,3.755098,0.764906
min,2.000000,1.000000,1.000000
25%,5.362500,6.000000,1.000000
50%,7.485000,8.000000,2.000000
75%,9.877500,10.750000,3.000000
max,15.660000,20.000000,3.000000


In [9]:
# rating distribution
employee_df["rating"].value_counts().sort_index()

,count
rating,
1,119
2,151
3,96


In [10]:
# rating proportions
employee_df["rating"].value_counts(normalize=True).sort_index()

,proportion
rating,
1,0.325137
2,0.412568
3,0.262295


In [11]:
# gender distribution
employee_df["gender"].value_counts()

,count
gender,
M,194
F,172


In [12]:
# region distribution
employee_df["region"].value_counts()

,count
region,
West,100
North,97
East,86
South,83


In [13]:
# performance summaries by rating
performance_by_rating = (
    employee_df
    .groupby("rating")[["sales", "new_customers"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

performance_by_rating

sales                                         new_customers             \
       count      mean median       std   min    max         count       mean   
rating                                                                          
1        119  5.339412   5.08  2.481580  2.00   9.92           119   5.067227   
2        151  7.987616   7.63  2.336428  4.03  12.09           151   8.490066   
3         96  9.575729   9.98  2.499215  5.08  15.66            96  10.958333   

                                 
       median       std min max  
rating                           
1         4.0  3.052454   1  10  
2         9.0  2.314786   5  12  
3        11.0  3.761066   3  20

In [14]:
# rating by gender
pd.crosstab(
    employee_df["gender"],
    employee_df["rating"],
    margins=True
)

rating,1,2,3,All
gender,,,,
F,52,69,51,172
M,67,82,45,194
All,119,151,96,366


In [15]:
# rating by region
pd.crosstab(
    employee_df["region"],
    employee_df["rating"],
    margins=True
)

rating,1,2,3,All
region,,,,
East,28,38,20,86
North,32,36,29,97
South,28,34,21,83
West,31,43,26,100
All,119,151,96,366


In [16]:
# correlation between numeric predictors
employee_df[["sales", "new_customers"]].corr()

,sales,new_customers
sales,1.000000,0.398721
new_customers,0.398721,1.000000


**Separate Hypothesis Tests for Each Variable**

**Assumption Checks for Initial Tests**

For ANOVA-style tests

Check:

*   approximate normality within each rating group
*   similar variances across rating groups
*   independence of observations

If assumptions are questionable, use Kruskal-Wallis.

For Chi-Square Tests

Check:

*   expected cell counts are sufficiently large
*   categories are mutually exclusive
*   observations are independent








**ANOVA assumption checks for sales across the three rating groups.**


Hypotheses:

*   H0: Sales are normally distributed within the group
*   HA: Sales are not normally distributed within the group



Conclusion: the normality assumption for ANOVA is **not** well satisfied.


In [17]:
# ANOVA assumptions checks for sales

# check normality of sales within each rating group using Shapiro-Wilk
for rating_group in sorted(employee_df["rating"].unique()):
    group_data = employee_df.loc[employee_df["rating"] == rating_group, "sales"]
    stat, p = stats.shapiro(group_data)

    print(f"Rating {rating_group}")
    print(f"Shapiro-Wilk statistic: {stat:.4f}")
    print(f"p-value: {p:.6f}")
    print()

Rating 1
Shapiro-Wilk statistic: 0.9165
p-value: 0.000002

Rating 2
Shapiro-Wilk statistic: 0.9418
p-value: 0.000007

Rating 3
Shapiro-Wilk statistic: 0.9668
p-value: 0.015612



**ANOVA assumption checks for sales across the three rating groups.**

**Levene’s test **checks whether the variance of sales is similar across the rating groups.

Hypotheses:

*   H0: The sales variances are equal across rating groups
*   HA: At least one group has a different variance


Your p-value is:

p=0.651861

Since this is greater than 0.05, we fail to reject the null.

So the equal variance assumption appears acceptable.

In [18]:
# ANOVA assumptions checks for sales

# check equal variances across rating groups using Levene's test
sales_groups = [
    employee_df.loc[employee_df["rating"] == rating_group, "sales"]
    for rating_group in sorted(employee_df["rating"].unique())
]

levene_stat, levene_p = stats.levene(*sales_groups)

print("Levene's Test for Equal Variances")
print(f"Statistic: {levene_stat:.4f}")
print(f"p-value: {levene_p:.6f}")

Levene's Test for Equal Variances
Statistic: 0.4284
p-value: 0.651861


**Practical conclusion for ANOVA assumptions**

*   For sales, the equal variance assumption looks fine, but the normality assumption fails. Because of that, we should probably use the **Kruskal-Wallis test** as the safer nonparametric alternative to **ANOVA**.

**Kruskal-Wallis test**

Kruskal-Wallis is the nonparametric alternative to one-way ANOVA.

*   Instead of comparing group means, it compares the rank distributions across groups.

*  So it is useful when the continuous variable is not normally distributed within groups.

* Kruskal-Wallis tells us that at least one rating group differs, but it does not identify exactly which rating groups differ from each other.


Do sales values differ significantly across rating groups 1, 2, and 3?

Hypotheses:

*   H0: The sales distributions are similar across rating groups
*   HA: At least one rating group has a different sales distribution

Conclusion:

*   Sales differ significantly across performance rating groups.

*   Employees with different performance ratings do not appear to have similar sales distributions. Sales are meaningfully associated with performance rating.



In [19]:
# Kruskal-Wallis test for sales across rating groups
sales_groups = [
    employee_df.loc[employee_df["rating"] == rating_group, "sales"]
    for rating_group in sorted(employee_df["rating"].unique())
]

kw_stat, kw_p = stats.kruskal(*sales_groups)

print("Kruskal-Wallis Test: Sales by Rating")
print(f"Statistic: {kw_stat:.4f}")
print(f"p-value: {kw_p:.6f}")

Kruskal-Wallis Test: Sales by Rating
Statistic: 108.4725
p-value: 0.000000


**ANOVA assumption checks for new_customers across the three rating groups.**


Hypotheses:

*   H0: new_customers are normally distributed within the group
*   HA: new_customers are not normally distributed within the group



Conclusion: the normality assumption for ANOVA is satisfied.

Practical conclusion for ANOVA assumptions

* For new customers, the normality assumption fail.

In [20]:
# ANOVA assumptions checks for new_customers
# check normality of new_customers within each rating group using Shapiro-Wilk
for rating_group in sorted(employee_df["rating"].unique()):
    group_data = employee_df.loc[employee_df["rating"] == rating_group, "new_customers"]
    stat, p = stats.shapiro(group_data)

    print(f"Rating {rating_group}")
    print(f"Shapiro-Wilk statistic: {stat:.4f}")
    print(f"p-value: {p:.6f}")
    print()

Rating 1
Shapiro-Wilk statistic: 0.9023
p-value: 0.000000

Rating 2
Shapiro-Wilk statistic: 0.9237
p-value: 0.000000

Rating 3
Shapiro-Wilk statistic: 0.9474
p-value: 0.000744



**Levene’s test** checks whether the variance of new_customers is similar across the rating groups.

Hypotheses:

*   H0: The new_customers variances are equal across rating groups
*   HA: At least one group has a different variance


Your p-value is:

p=0.000968

Since this is less than 0.05, we reject the null.

So the equal variance assumption is violated.

In [21]:
# ANOVA assumptions checks for new_customers
# check equal variances across rating groups using Levene's test
new_customer_groups = [
    employee_df.loc[employee_df["rating"] == rating_group, "new_customers"]
    for rating_group in sorted(employee_df["rating"].unique())
]

levene_stat, levene_p = stats.levene(*new_customer_groups)

print("Levene's Test for Equal Variances")
print(f"Statistic: {levene_stat:.4f}")
print(f"p-value: {levene_p:.6f}")

Levene's Test for Equal Variances
Statistic: 7.0744
p-value: 0.000968


****Kruskal-Wallis test****

Do new_customers values differ significantly across rating groups 1, 2, and 3?

Hypotheses:

*   H0: The new_customers distributions are similar across rating groups
*   HA: At least one rating group has a different new_customers distribution

Conclusion:
* We reject the null hypothesis.
* New customer counts differ significantly across performance rating groups.

In practical terms:

* Employees with different performance ratings do not have similar new-customer distributions. New-customer acquisition appears meaningfully associated with performance rating.

Combined with the new_customers result, this is good early evidence that the rating system is connected to job-relevant performance measures.

In [22]:
# Kruskal-Wallis test for new customers across rating groups
new_customer_groups = [
    employee_df.loc[employee_df["rating"] == rating_group, "new_customers"]
    for rating_group in sorted(employee_df["rating"].unique())
]

kw_stat, kw_p = stats.kruskal(*new_customer_groups)

print("Kruskal-Wallis Test: New Customers by Rating")
print(f"Statistic: {kw_stat:.4f}")
print(f"p-value: {kw_p:.6f}")

Kruskal-Wallis Test: New Customers by Rating
Statistic: 128.7708
p-value: 0.000000


**Diving deeper beyond the Kruskal-Wallis test.**

**Kruskal-Wallis question**: Are all rating groups similar, or does at least one rating group differ?

* Is there evidence that the distribution of sales differs across rating groups 1, 2, and 3?

* Is there evidence that the distribution of new_customers differs across rating groups 1, 2, and 3?

**Dunn’s post-hoc question**

If Kruskal-Wallis is significant, Dunn’s test asks:
* Which specific pairs of rating groups differ from each other?

For example:

* Rating 1 vs Rating 2
* Rating 1 vs Rating 3
* Rating 2 vs Rating 3

**Bonferroni correction purpose**

Because Dunn’s test performs multiple pairwise comparisons, the Bonferroni correction adjusts the p-values to reduce the chance of false positives.

**Sales Dunn’s post-hoc test**

* All pairwise comparisons are statistically significant after Bonferroni correction:

| Comparison    |       Adjusted p-value | Conclusion  |
| ------------- | ---------------------: | ----------- |
| Rating 1 vs 2 | (5.86 \times 10^{-12}) | significant |
| Rating 1 vs 3 | (8.35 \times 10^{-24}) | significant |
| Rating 2 vs 3 |  (1.37 \times 10^{-4}) | significant |

So for sales, every rating group differs from every other rating group.

Practical interpretation:

* Sales increase meaningfully across rating levels. Employees rated 1, 2, and 3 have significantly different sales distributions, which supports the idea that sales are strongly connected to performance ratings.

In [23]:
# Dunn's post-hoc test for sales by rating with Bonferroni correction
dunn_sales = sp.posthoc_dunn(
    employee_df,
    val_col="sales",
    group_col="rating",
    p_adjust="bonferroni"
)

dunn_sales

,1,2,3
1,1.000000e+00,5.859373e-12,8.354898e-24
2,5.859373e-12,1.000000e+00,1.373892e-04
3,8.354898e-24,1.373892e-04,1.000000e+00


**New customers Dunn’s post-hoc test**

Again, all pairwise comparisons are statistically significant after Bonferroni correction:

| Comparison    |       Adjusted p-value | Conclusion  |
| ------------- | ---------------------: | ----------- |
| Rating 1 vs 2 | (1.81 \times 10^{-13}) | significant |
| Rating 1 vs 3 | (2.93 \times 10^{-28}) | significant |
| Rating 2 vs 3 |  (1.05 \times 10^{-5}) | significant |

So for new_customers, every rating group also differs from every other rating group.

Practical interpretation:

* New customer acquisition increases meaningfully across rating levels. Employees with ratings 1, 2, and 3 have significantly different new-customer distributions.

In [24]:
# Dunn's post-hoc test for new customers by rating with Bonferroni correction
dunn_new_customers = sp.posthoc_dunn(
    employee_df,
    val_col="new_customers",
    group_col="rating",
    p_adjust="bonferroni"
)

dunn_new_customers

,1,2,3
1,1.000000e+00,1.812187e-13,2.929544e-28
2,1.812187e-13,1.000000e+00,1.048262e-05
3,2.929544e-28,1.048262e-05,1.000000e+00


**chi-square test** of independence for the categorical predictors.

* rating vs region
* rating vs gender

Setup the table to test whether the distribution of performance ratings differs across regions.

In [25]:
# contingency table: rating by region
region_rating_table = pd.crosstab(
    employee_df["region"],
    employee_df["rating"]
)

region_rating_table

rating,1,2,3
region,,,
East,28,38,20
North,32,36,29
South,28,34,21
West,31,43,26


Test whether the distribution of performance ratings differs across region.

Hypotheses:

* H0: Rating distribution is independent of region
* HA: Rating distribution differs by region

Conclusion
* p=0.9533

* Since this is much greater than 0.05, we fail to reject the null hypothesis.

Interpretation

* There is no evidence that the distribution of performance ratings differs by region. In practical terms, employees appear to receive similar rating distributions across North, South, East, and West.

In [26]:
# chi-square test of independence: rating vs region
chi2, p, dof, expected = stats.chi2_contingency(region_rating_table)

print("Chi-square Test: Rating by Region")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(
    expected,
    index=region_rating_table.index,
    columns=region_rating_table.columns
)

expected_df

Chi-square Test: Rating by Region
Chi-square statistic: 1.5898
p-value: 0.953305
Degrees of freedom: 6


rating,1,2,3
region,,,
East,27.961749,35.480874,22.557377
North,31.538251,40.019126,25.442623
South,26.986339,34.243169,21.770492
West,32.513661,41.256831,26.229508


Setup the table to test whether the distribution of performance ratings differs by gender.

In [27]:
# contingency table: rating by gender
gender_rating_table = pd.crosstab(
    employee_df["gender"],
    employee_df["rating"]
)

gender_rating_table

rating,1,2,3
gender,,,
F,52,69,51
M,67,82,45


Test whether the distribution of performance ratings differs across gender.

Hypotheses:

* H0: Rating distribution is independent of gender
* HA: Rating distribution differs by gender


Conclusion

* p=0.3552

* Since p>0.05, we fail to reject the null hypothesis.

Interpretation

* There is no statistically significant evidence that performance rating distribution differs by gender. In practical terms, males and females appear to receive similar distributions of ratings in this unadjusted categorical test.

This is a good preliminary fairness signal for gender, but we still need the ordered logistic model to check whether gender matters after controlling for sales, new customers, and region.

In [28]:
# chi-square test of independence: rating vs gender
chi2, p, dof, expected = stats.chi2_contingency(gender_rating_table)

print("Chi-square Test: Rating by Gender")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(
    expected,
    index=gender_rating_table.index,
    columns=gender_rating_table.columns
)

expected_df

Chi-square Test: Rating by Gender
Chi-square statistic: 2.0700
p-value: 0.355220
Degrees of freedom: 2


rating,1,2,3
gender,,,
F,55.923497,70.961749,45.114754
M,63.076503,80.038251,50.885246


**Main Model Selection**

The appropriate choice is **ordered logistic regression** because **rating** is an **ordinal outcome**.

Before fitting, the only quick remaining check to do is a light pre-model diagnostic review (mainly correlation/VIF and category coding), then fit the ordered logit model.

**Pre-Model Diagnostics**

In [29]:
# confirm unique category values
print("rating values:", sorted(employee_df["rating"].unique()))
print("gender values:", employee_df["gender"].unique())
print("region values:", employee_df["region"].unique())

rating values: [np.int64(1), np.int64(2), np.int64(3)]
gender values: ['M' 'F']
region values: ['North' 'West' 'South' 'East']


In [30]:
# correlation between numeric predictors
employee_df[["sales", "new_customers"]].corr()

,sales,new_customers
sales,1.000000,0.398721
new_customers,0.398721,1.000000


In [31]:
# create dummy-coded design matrix for VIF check
vif_data = pd.get_dummies(
    employee_df[["sales", "new_customers", "region", "gender"]],
    columns=["region", "gender"],
    drop_first=True,
    dtype=float
)

vif_data.head()

,sales,new_customers,region_North,region_South,region_West,gender_M
0,10.10,9,1.0,0.0,0.0,1.0
1,11.67,17,0.0,0.0,1.0,0.0
2,8.91,13,1.0,0.0,0.0,1.0
3,7.73,10,0.0,1.0,0.0,0.0
4,11.36,12,1.0,0.0,0.0,1.0


**Variance Inflation Factor** (VIF)

* VIF measures how much the uncertainty of a coefficient is inflated because that predictor overlaps with the other predictors in the model.

|  VIF | Meaning                            |
| ---: | ---------------------------------- |
|   ~1 | no multicollinearity concern       |
|  1–5 | generally acceptable               |
| 5–10 | moderate concern / inspect further |
|  >10 | serious multicollinearity concern  |


**Variance Inflation Factor** Analysis
* There is no severe multicollinearity, but sales and new_customers show moderate overlap and should be monitored.

* This makes sense because they are both performance measures, and their correlation is about 0.40. It is not bad enough to drop either variable, especially because both are substantively important.

In [32]:
# calculate VIF values
vif_df = pd.DataFrame()

vif_df["Variable"] = vif_data.columns

vif_df["VIF"] = [
    variance_inflation_factor(vif_data.values, i)
    for i in range(vif_data.shape[1])
]

vif_df

,Variable,VIF
0,sales,6.717922
1,new_customers,6.060891
2,region_North,1.784792
3,region_South,1.704914
4,region_West,1.741467
5,gender_M,1.897136


**Informal Proportional Odds Check**

The **proportional odds assumption** checks whether it is reasonable to treat the rating scale as one ordered progression, rather than as separate category transitions with different predictor effects.

The proportional odds assumption says the effect of sales, for example, is consistent across both comparisons.

So if sales helps distinguish:

* 1 vs 2/3

then it should have the same kind of relationship when distinguishing:

* 1/2 vs 3

The ordered logistic model uses one coefficient for sales, not separate coefficients for each cutoff.

Conclusion: **Proportional Odds Check**

The **proportional odds assumption** looks reasonably acceptable for the main performance variables sales and new_customers.

* The coefficients are not identical, especially for sales, but they are not wildly different or reversing direction.

* This suggests the ordered logistic model’s single-coefficient simplification is defensible for our purposes.

* For gender, the sign changes from slightly positive to slightly negative, but both coefficients are statistically insignificant. So I would not treat that as strong evidence against the model.

* For region, there is some movement, especially for North, but the region effects are also not statistically significant in either threshold model.

Practical conclusion
* The proportional odds assumption appears reasonably plausible for the key predictors. sales and new_customers consistently increase the odds of being in a higher rating category across both rating thresholds. Therefore, the ordered logistic model is a reasonable choice for this analysis.

In [33]:
# create binary outcomes for each ordered threshold
employee_df["rating_gt_1"] = (employee_df["rating"] > 1).astype(int)
employee_df["rating_gt_2"] = (employee_df["rating"] > 2).astype(int)

# threshold model 1: rating 2 or 3 versus rating 1
logit_gt_1 = smf.logit(
    formula="rating_gt_1 ~ sales + new_customers + C(region) + C(gender)",
    data=employee_df
).fit()

# threshold model 2: rating 3 versus rating 1 or 2
logit_gt_2 = smf.logit(
    formula="rating_gt_2 ~ sales + new_customers + C(region) + C(gender)",
    data=employee_df
).fit()

print(logit_gt_1.summary())
print(logit_gt_2.summary())

Optimization terminated successfully.
         Current function value: 0.359061
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.406331
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:            rating_gt_1   No. Observations:                  366
Model:                          Logit   Df Residuals:                      359
Method:                           MLE   Df Model:                            6
Date:                Fri, 15 May 2026   Pseudo R-squ.:                  0.4307
Time:                        19:21:51   Log-Likelihood:                -131.42
converged:                       True   LL-Null:                       -230.83
Covariance Type:            nonrobust   LLR p-value:                 3.374e-40
                         coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

In [34]:
# compare coefficients across the two threshold models
po_check = pd.DataFrame({
    "rating_gt_1_coef": logit_gt_1.params,
    "rating_gt_2_coef": logit_gt_2.params
})

po_check["difference"] = po_check["rating_gt_2_coef"] - po_check["rating_gt_1_coef"]

po_check

,rating_gt_1_coef,rating_gt_2_coef,difference
Intercept,-5.908253,-6.742421,-0.834168
C(region)[T.North],-0.162737,0.448562,0.611299
C(region)[T.South],0.029017,0.197684,0.168667
C(region)[T.West],0.360602,0.326867,-0.033736
C(gender)[T.M],0.166210,-0.147440,-0.313649
sales,0.495891,0.315719,-0.180171
new_customers,0.427690,0.323082,-0.104608


**Ordered Logistic Regression Model**

**Conclusion**: Sales and New Customers

Both `sales` and `new_customers` are statistically significant predictors of performance rating. As expected, these performance-related variables appear to be the primary drivers of the rating outcome.

Holding all other variables constant, each additional million dollars in sales is associated with approximately 47% higher odds of being in a higher performance rating category rather than a lower one.

Holding all other variables constant, each additional new customer acquired is associated with approximately 47% higher odds of being in a higher performance rating category rather than a lower one.

**Conclusion**: Gender

**chi-square test of independence** preliminary analysis
* Preliminary gender analysis used the chi-square test of independence between gender and rating.
* There was no statistically significant evidence that performance ratings differed by gender in the raw, unadjusted categorical comparison.



The **ordered logistic model **lets us ask the stronger adjusted question:

* After controlling for sales, new_customers, and region, is gender still associated with receiving a higher performance rating?


There is no strong statistical evidence of gender bias in rating outcomes based on this model.

* The coefficient for gender_M is slightly negative, indicating that male salespersons are associated with slightly lower odds of being in a higher performance rating category compared with female salespersons, holding sales, new customers, and region constant.

* The odds ratio is approximately 0.978, which corresponds to about a 2.2% decrease in the odds of being in a higher rating category. However, this result is not statistically significant, so there is no strong evidence that gender is associated with performance ratings after accounting for measurable performance and region.

In [35]:
# ordered logistic regression model

# create modeling dataframe
ordered_model_df = employee_df.copy()

# ensure rating is coded as ordered integer outcome
ordered_model_df["rating"] = ordered_model_df["rating"].astype(int)

# dummy-code categorical predictors
ordered_model_df = pd.get_dummies(
    ordered_model_df,
    columns=["region", "gender"],
    drop_first=True,
    dtype=float
)

# define outcome and predictors
y = ordered_model_df["rating"]

X = ordered_model_df[
    [
        "sales",
        "new_customers",
        "region_North",
        "region_South",
        "region_West",
        "gender_M"
    ]
]

# fit ordered logistic regression model
ordered_logit_model = OrderedModel(
    y,
    X,
    distr="logit"
)

ordered_logit_results = ordered_logit_model.fit(method="bfgs")

# display results in statsmodels-style summary format
print(ordered_logit_results.summary())

Optimization terminated successfully.
         Current function value: 0.737981
         Iterations: 36
         Function evaluations: 38
         Gradient evaluations: 38
                             OrderedModel Results                             
Dep. Variable:                 rating   Log-Likelihood:                -270.10
Model:                   OrderedModel   AIC:                             556.2
Method:            Maximum Likelihood   BIC:                             587.4
Date:                Fri, 15 May 2026                                         
Time:                        19:21:52                                         
No. Observations:                 366                                         
Df Residuals:                     358                                         
Df Model:                           6                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------

In [36]:
# create clean ordered logit results table
ordered_results_table = pd.DataFrame({
    "coef": ordered_logit_results.params,
    "std_err": ordered_logit_results.bse,
    "z": ordered_logit_results.tvalues,
    "p_value": ordered_logit_results.pvalues,
    "ci_lower": ordered_logit_results.conf_int()[0],
    "ci_upper": ordered_logit_results.conf_int()[1],
})

# add odds ratios for non-threshold coefficients
ordered_results_table["odds_ratio"] = np.exp(ordered_results_table["coef"])
ordered_results_table["or_ci_lower"] = np.exp(ordered_results_table["ci_lower"])
ordered_results_table["or_ci_upper"] = np.exp(ordered_results_table["ci_upper"])

# round for readability
ordered_results_table = ordered_results_table.round({
    "coef": 4,
    "std_err": 4,
    "z": 3,
    "p_value": 4,
    "ci_lower": 4,
    "ci_upper": 4,
    "odds_ratio": 3,
    "or_ci_lower": 3,
    "or_ci_upper": 3
})

ordered_results_table

,coef,std_err,z,p_value,ci_lower,ci_upper,odds_ratio,or_ci_lower,or_ci_upper
sales,0.3851,0.0477,8.072,0.0000,0.2916,0.4786,1.470,1.339,1.614
new_customers,0.3867,0.0413,9.373,0.0000,0.3058,0.4675,1.472,1.358,1.596
region_North,0.1821,0.3225,0.565,0.5724,-0.4500,0.8141,1.200,0.638,2.257
region_South,0.0945,0.3355,0.282,0.7783,-0.5631,0.7520,1.099,0.569,2.121
region_West,0.3033,0.3215,0.943,0.3454,-0.3268,0.9333,1.354,0.721,2.543
gender_M,-0.0222,0.2300,-0.097,0.9231,-0.4730,0.4286,0.978,0.623,1.535
1/2,4.9350,0.5431,9.086,0.0000,3.8704,5.9995,139.067,47.963,403.224
2/3,1.1028,0.0768,14.355,0.0000,0.9522,1.2534,3.013,2.591,3.502


**Model Diagnostics Assumptions**

## Model Diagnostics and Assumptions

The ordered logistic regression model converged successfully, indicating that the estimation procedure was stable. Prior to fitting the model, multicollinearity was assessed using VIF values. `sales` and `new_customers` showed moderate overlap, which is expected because both are performance-related measures, but the VIF values were not severe enough to justify removing either variable.

The main assumption for ordered logistic regression is the proportional odds assumption, which states that each predictor has a consistent effect across the thresholds separating the ordered rating categories.

In this analysis, the model results are substantively reasonable and consistent with the earlier hypothesis tests: `sales` and `new_customers` are significant positive predictors, while `region` and `gender` are not statistically significant.

If the proportional odds assumption were seriously questionable, reasonable alternatives would include a multinomial logistic regression, a partial proportional odds model, or a sensitivity analysis comparing results across model specifications.

**Final Conclusions and Recommendations**

**Executive Summary**

The analysis suggests that employee performance ratings are strongly aligned with measurable job performance. Both sales and new_customers differed significantly across rating groups, and Dunn’s post-hoc tests showed that all three rating levels were meaningfully distinct from one another on both measures.

The ordered logistic regression model confirmed these findings. Holding other variables constant, both higher sales and more new customers were statistically significant predictors of receiving a higher performance rating. Each additional million dollars in sales was associated with approximately 47% higher odds of being in a higher rating category, and each additional new customer was also associated with approximately 47% higher odds of being in a higher rating category.

There was no strong evidence that gender or region influenced ratings. The chi-square tests showed no significant differences in rating distributions by gender or region, and neither variable was statistically significant in the ordered logistic regression model after controlling for sales and new customers.

Overall, the performance evaluation process appears to be largely data-driven and tied to job-relevant outcomes. The proportional odds diagnostic was also reasonably supportive of the ordered logistic model, especially for the key performance predictors. Based on this analysis, there is no strong statistical evidence of unfairness by gender or unexplained regional rating differences, though any final organizational conclusion should still consider qualitative context about territory difficulty, manager judgment, and rating procedures.